# Preprocessing: Coregistration Test
이 노트북은 K3A 영상의 RPC 정사보정 및 2.5m 가상 격자 투영이 정상적으로 동작하는지 테스트합니다.

In [ ]:
import sys
import os

# [중요] PostGIS와 Anaconda의 PROJ 버전 충돌 방지 코드
# rasterio 패키지 내부에 번들로 포함된 proj_data 폴더를 참조하도록 강제합니다.
import rasterio
proj_path = os.path.join(os.path.dirname(rasterio.__file__), 'proj_data')
os.environ['PROJ_LIB'] = proj_path
os.environ['PROJ_DATA'] = proj_path

sys.path.append('../..')  # 프로젝트 루트를 sys.path 에 추가 (module2_coregistration 패키지 import 용)

from rasterio.plot import show
import matplotlib.pyplot as plt
from pathlib import Path

from module2_coregistration.src import create_virtual_grid, orthorectify_k3a_with_rpc

## 1. 데이터 경로 설정 및 원본 바운딩 박스 추출

In [ ]:
k3a_tif_path = Path('../data/interim/k3a_extracted/K3A_20150401044329_00095_00004060_L1R/K3A_20150401044329_00095_00004060_L1R_B.tif')
out_tif_path = Path('../data/interim/test_k3a_ortho_B.tif')

# K3A 원본 바운딩 박스 확인
with rasterio.open(k3a_tif_path) as src:
    k3a_bounds = src.bounds
    print("Original K3A Bounds:", k3a_bounds)
    print("Original CRS:", src.crs)
    
# S2 영상은 아직 없으므로, 교집합을 위해 K3A bounds를 그대로 사용 (또는 임의로 약간 축소)
s2_dummy_bounds = k3a_bounds

## 2. 가상 격자(Virtual Grid) 프로필 생성

In [ ]:
# UTM Zone 52N (EPSG:32652) 기준으로 2.5m 격자 생성
# K3A 원본 CRS가 EPSG:4326(경위도)라면 먼저 투영 변환이 필요할 수 있으나, rasterio reproject가 알아서 처리합니다.
grid_profile = create_virtual_grid(
    k3a_bounds=k3a_bounds,
    s2_bounds=s2_dummy_bounds,
    resolution=2.5,
    crs="EPSG:32652"
)
print("Virtual Grid Profile:")
print(grid_profile)

## 3. RPC 기반 정사보정 및 투영 (Orthorectification)

In [ ]:
# 만약 rasterio가 _rpc.txt를 자동으로 읽지 못할 경우 에러가 발생할 수 있습니다.
try:
    orthorectify_k3a_with_rpc(
        k3a_tif_path=k3a_tif_path,
        grid_profile=grid_profile,
        out_path=out_tif_path
    )
    print("Orthorectification Successful!")
except Exception as e:
    print("Error during orthorectification:", e)
    print("\n[TIP] Rasterio가 소문자 _rpc.txt를 인식하지 못했을 가능성이 높습니다.")
    print("원본 폴더의 _rpc.txt 파일 이름을 _RPC.TXT (대문자)로 변경한 후 다시 실행해 보세요.")

## 4. 결과 시각화 확인

In [ ]:
if out_tif_path.exists():
    with rasterio.open(out_tif_path) as src:
        fig, ax = plt.subplots(figsize=(10, 10))
        # 히스토그램 평활화 등 시각화 개선을 위해 간단히 1% ~ 99% 스트레칭 후 출력할 수도 있음
        show(src, ax=ax, cmap='gray', title="Orthorectified K3A (B Band) - 2.5m")
else:
    print("Result file does not exist.")